# Pothole & Road Damage Detector: run the website in Colab (T4 GPU)

This notebook starts the **Flask website** from the project in Colab, using the **pretrained weights** (`final.keras`). No training.
Upload a road video on the web page and get a pothole verdict plus an annotated video.

Before you start: `Runtime > Change runtime type > T4 GPU`, then run the cells top to bottom (Shift+Enter).
Step-by-step guide: `COLAB_SETUP.md` in the repo.

## 1. Check the GPU

In [ ]:
import tensorflow as tf
!nvidia-smi -L
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
assert gpus, "No GPU: Runtime > Change runtime type > T4 GPU, then reconnect and re-run."

## 2. Get the project code
Clones the repo (the repo is private, so paste a read-only GitHub token; leave empty if it is public).
If you already uploaded the project folder to `/content/pothole-cnn`, this step is skipped.

In [ ]:
import os, getpass, subprocess
REPO = "mourishantony/pothole-cnn"
DEST = "/content/pothole-cnn"
if not os.path.exists(DEST):
    token = getpass.getpass("GitHub token (empty if public): ").strip()
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    res = subprocess.run(["git", "clone", "-q", url, DEST], capture_output=True, text=True,
                         env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    err = res.stderr.replace(token, "***") if token else res.stderr
    del token, url
    if res.returncode != 0:
        print(err.strip())
        print()
        print("Clone failed, nothing was downloaded. Fix one of these, then re-run this cell:")
        print(" 1) Fine-grained token: Repository access = pothole-cnn, Contents = Read-only (or use a classic token with 'repo' scope)")
        print(" 2) Make the repo public and press Enter at the prompt")
        print(" 3) Upload the project zip instead: see COLAB_SETUP.md, option C")
        raise SystemExit("git clone failed")
    subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"])  # do not keep the token on disk
%cd /content/pothole-cnn
!ls


## 3. Add the weights file (`final.keras`)
The model file must be at **`/content/pothole-cnn/models/final.keras`** (about 83 MB).
- If the clone above included it, this cell just confirms it.
- Otherwise: either drag `final.keras` into the `pothole-cnn/models/` folder in Colab's **Files** panel (folder icon on the left),
  or run this cell and pick the file when asked.

In [ ]:
import os, shutil
WEIGHTS = "/content/pothole-cnn/models/final.keras"

def weights_ok():
    return os.path.exists(WEIGHTS) and os.path.getsize(WEIGHTS) > 50e6

if weights_ok():
    print(f"Weights found: {os.path.getsize(WEIGHTS)/1e6:.0f} MB")
else:
    from google.colab import files
    print("Weights not found: choose final.keras (about 83 MB) from your computer")
    up = files.upload()
    os.makedirs(os.path.dirname(WEIGHTS), exist_ok=True)
    shutil.move(next(iter(up)), WEIGHTS)
assert weights_ok(), "final.keras is missing or too small (expected ~83 MB). Upload the correct file."
assert os.path.exists("/content/pothole-cnn/models/classes.json"), "models/classes.json is missing"
print("Ready:", WEIGHTS)

## 4. Install the extra packages
Colab already has TensorFlow and OpenCV. Only Flask and ffmpeg are added.

In [ ]:
!pip install -q flask imageio-ffmpeg

## 5. Load the model
Expected output: `Model loaded. Classes: ['crack', 'normal', 'pothole']`.
If it fails with `could not be deserialized` / `input_axes`, run `!pip install -U keras` then `Runtime > Restart session` and re-run from step 1.

In [ ]:
import sys
sys.path.insert(0, "/content/pothole-cnn")
import app
print("Model loaded. Classes:", app.CLASSES)

## 6. Start the website
The web page appears below the cell. Choose a road video (mp4/avi/mov/mkv/webm, up to 300 MB) and click **Analyse video**.
Red boxes = pothole, orange = crack. Keep this cell's output open while you use the page.
To restart the site, use `Runtime > Restart session` (the port stays busy otherwise).

In [ ]:
import threading
from google.colab import output

threading.Thread(
    target=lambda: app.app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False, threaded=True),
    daemon=True,
).start()
output.serve_kernel_port_as_iframe(5000, height=850)

## Fallback: analyse a video without the web page
Only if uploads through the embedded page fail: set `RUN_FALLBACK = True` in the cell below and run it.
It is off by default so that "Run all" does not stop here waiting for a file.


In [ ]:
RUN_FALLBACK = False   # set to True only if uploads through the web page fail (so "Run all" does not stop here)

if RUN_FALLBACK:
    import pathlib
    from google.colab import files
    from IPython.display import Video, Image, display

    uploaded = files.upload()
    video_path = next(iter(uploaded))
    out_dir = pathlib.Path("/content/results"); out_dir.mkdir(exist_ok=True)
    r = app.process(pathlib.Path(video_path), out_dir, every_sec=0.5)
    print(r)
    print("==> POTHOLE DETECTED" if r["pothole_found"] else "==> No pothole detected")
    display(Video(str(out_dir / "annotated.mp4"), embed=True, width=720))
    if r["has_best"]:
        display(Image(str(out_dir / "best.jpg")))
else:
    print("Fallback skipped. Use the web page from the 'Start the website' cell above.")
